# Gaussian bandwidth-choice distributions: L² and ISE

Compare **L² spectral selection and the sample-specific ISE choice** on the same repeated samples from $N(0,1)$ and the same geometric bandwidth grid.
For spectral selection, use $S_{ij}=\exp(-\lambda D_{ij})$ with **$\lambda=1$** and select the bandwidth at the largest Perron-vector component.
The ISE choice minimizes exact integrated squared error over the same candidate grid.

The L² divergence is $D(p,q)=\frac12\int(p-q)^2$, evaluated with exact Gaussian KDE inner products. The factor $1/2$ follows the bandwidth-distribution experiment in the thesis.
Sources: [thesis beta-2 formula, `eq:beta2_exact_formula01`, and Figure 3.5](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [shared exact Gaussian implementation](kde_tools/similarities.py); [exact ISE benchmark](kde_tools/density.py).

Run all cells. The **preview** profile produces a small comparison; **thesis** uses the original Figure 3.5 sample/repetition/grid settings. A checkpoint saves each completed repetition, so rerunning resumes the same configuration.
Spectral choices use the centralized stabilized Lanczos solver in `kde_tools.perron`; checkpoint settings record its version and the two retained methods.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
import time
from pathlib import Path

working_directory = Path.cwd().resolve()
code_directory = next(
    (candidate.resolve()
     for root in (working_directory, *working_directory.parents)
     for candidate in (root, root / "Code" / "Code_kde")
     if (candidate / "kde_tools" / "pdf_export.py").is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from the project or Code/Code_kde folder.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(code_directory / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import PercentFormatter

from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, gaussian_mise_oracle_bandwidth
from kde_tools.density import exact_oracle_ise_values
from kde_tools.perron import PERRON_SOLVER_VERSION
from kde_tools.pdf_export import enable_automatic_pdf_saving
from kde_tools.similarities import beta2_divergence_matrix, spectral_choice_from_divergence

notebook_name = "gaussian_symmetric_divergence_bandwidth_distributions"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name, output_root=code_directory / "outputs" / "pdfs"
)
data_output_directory = code_directory / "outputs" / "data" / notebook_name
image_output_directory = code_directory / "outputs" / "images" / notebook_name
for directory in (data_output_directory, image_output_directory):
    directory.mkdir(parents=True, exist_ok=True)
print(f"Automatic PDF exports: {pdf_output_directory}")


## Settings

Both profiles draw $n=200$ observations from $N(0,1)$ and use $\lambda=1$, $h_{\min}=0.01$, $h_{\max}=2$. The preview uses $M=20$, $N=21$; the thesis profile uses $M=1000$, $N=201$, matching the [Figure 3.5 caption](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex).
All settings are editable below. L² and ISE use exact Gaussian formulas, so no spatial integration grid is required; see [similarities.py](kde_tools/similarities.py) and [density.py](kde_tools/density.py).


In [ ]:
RUN_PROFILE = "thesis"  # "preview" or "thesis"
PROFILES = {
    "preview": {"sample_size": 200, "repetitions": 20, "grid_size": 21},
    "thesis": {"sample_size": 200, "repetitions": 1000, "grid_size": 201},
}
settings = {
    "perron_solver": PERRON_SOLVER_VERSION,
    **PROFILES[RUN_PROFILE],
    "sigma2": 1.0,
    "lambda_value": 1.0,
    "h_min": 0.01,
    "h_max": 2.0,
    "base_seed": 910_000,
    "implementation_version": 3,
}
reuse_checkpoint = True
show_mise_reference = True

if settings["sample_size"] < 1 or settings["repetitions"] < 1:
    raise ValueError("sample_size and repetitions must be positive.")
if settings["sigma2"] <= 0 or settings["lambda_value"] <= 0:
    raise ValueError("sigma2 and lambda_value must be positive.")
h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=settings["h_min"], h_max=settings["h_max"],
    beta=2.0, grid_size=settings["grid_size"],
)
methods = ("L2", "ISE")
configuration_json = json.dumps({"settings": settings, "methods": methods}, sort_keys=True)
configuration_id = hashlib.sha256(configuration_json.encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
run_directory.mkdir(parents=True, exist_ok=True)
checkpoint_path = run_directory / "checkpoint.npz"
plot_stem = f"{RUN_PROFILE}_l2_ise_n{settings['sample_size']}_M{settings['repetitions']}_N{len(h_grid)}"
print(f"Profile: {RUN_PROFILE}; n={settings['sample_size']}, M={settings['repetitions']}, N={len(h_grid)}, lambda={settings['lambda_value']:g}")
print(f"Checkpoint and data: {run_directory}")


## One repetition: L² spectral selection and an ISE choice

The same sample supplies both choices. L² uses the shared exact divergence and Perron selector; the ISE choice minimizes the exact squared error on the same bandwidth grid.
Sources: [shared spectral selection](kde_tools/similarities.py), [exact ISE helper](kde_tools/density.py).


In [ ]:
def select_one_sample(samples, h_grid, settings):
    divergence = beta2_divergence_matrix(samples, h_grid)
    selections = {"L2": int(spectral_choice_from_divergence(
        divergence, settings["lambda_value"]
    )["selected_index"])}
    ise_values = exact_oracle_ise_values(samples, h_grid, settings["sigma2"])
    selections["ISE"] = int(np.argmin(ise_values))
    selected_ise = {name: float(ise_values[index]) for name, index in selections.items()}
    return selections, selected_ise


## Repeated sampling, with resumable checkpoints

Each repetition uses seed `base_seed + repetition`, independently of which methods are computed. The checkpoint contains the settings and all completed L² and ISE choices. Changing numerical settings creates a new run directory. Set `reuse_checkpoint=False` to recompute the current configuration.

In [ ]:
repetitions = settings["repetitions"]
selected_indices = np.full((repetitions, len(methods)), -1, dtype=int)
selected_ise_values = np.full((repetitions, len(methods)), np.nan)
completed = 0
if reuse_checkpoint and checkpoint_path.exists():
    with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
        if str(checkpoint["configuration_json"].item()) != configuration_json:
            raise ValueError("Checkpoint settings differ from the current configuration.")
        completed = int(checkpoint["completed"].item())
        selected_indices[:] = checkpoint["selected_indices"]
        selected_ise_values[:] = checkpoint["selected_ise_values"]
    if not 0 <= completed <= repetitions or np.any(selected_indices[:completed] < 0):
        raise ValueError("Checkpoint has invalid completed selections.")
    print(f"Resuming after {completed}/{repetitions} completed repetitions.")

started = time.perf_counter()
for repetition in range(completed, repetitions):
    rng = np.random.default_rng(settings["base_seed"] + repetition)
    samples = rng.normal(0.0, np.sqrt(settings["sigma2"]), size=settings["sample_size"])
    choices, ise_values = select_one_sample(samples, h_grid, settings)
    selected_indices[repetition] = [choices[name] for name in methods]
    selected_ise_values[repetition] = [ise_values[name] for name in methods]
    temporary_checkpoint = checkpoint_path.with_suffix(".npz.tmp")
    with temporary_checkpoint.open("wb") as stream:
        np.savez_compressed(
            stream, configuration_json=configuration_json, completed=repetition + 1,
            selected_indices=selected_indices, selected_ise_values=selected_ise_values,
        )
    temporary_checkpoint.replace(checkpoint_path)
    print(f"Completed {repetition + 1}/{repetitions}; elapsed this run: {time.perf_counter() - started:.1f}s", flush=True)

counts = {name: np.bincount(selected_indices[:, column], minlength=len(h_grid))
          for column, name in enumerate(methods)}
frequencies = {name: values / repetitions for name, values in counts.items()}
for name in methods:
    assert counts[name].sum() == repetitions
    assert np.isclose(frequencies[name].sum(), 1.0)
assert np.all(selected_ise_values[:, -1, None] <= selected_ise_values + 1e-12)


## Summary and saved data

The table reports selected bandwidths and the exact ISE evaluated at each choice. Export the individual choices, frequency table, summary, and settings so that the plotted distributions can be reproduced without rerunning the simulations.

In [ ]:
def write_csv(path, fieldnames, rows):
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

selection_rows = [
    {"repetition": repetition, "seed": settings["base_seed"] + repetition,
     "method": name, "grid_index": int(selected_indices[repetition, column]),
     "h": float(h_grid[selected_indices[repetition, column]]),
     "ise": float(selected_ise_values[repetition, column])}
    for repetition in range(repetitions) for column, name in enumerate(methods)
]
frequency_rows = [
    {"method": name, "grid_index": index, "h": float(h_grid[index]),
     "count": int(counts[name][index]), "relative_frequency": float(frequencies[name][index])}
    for name in methods for index in range(len(h_grid))
]
summary_rows = []
for column, name in enumerate(methods):
    selected_h = h_grid[selected_indices[:, column]]
    modal_index = int(np.argmax(counts[name]))
    summary_rows.append({
        "method": name, "modal_h": float(h_grid[modal_index]),
        "mean_h": float(np.mean(selected_h)), "median_h": float(np.median(selected_h)),
        "modal_frequency": float(frequencies[name][modal_index]),
        "mean_ise": float(np.mean(selected_ise_values[:, column])),
    })
write_csv(run_directory / "selections.csv", selection_rows[0].keys(), selection_rows)
write_csv(run_directory / "frequencies.csv", frequency_rows[0].keys(), frequency_rows)
write_csv(run_directory / "summary.csv", summary_rows[0].keys(), summary_rows)
(run_directory / "settings.json").write_text(json.dumps({
    "profile": RUN_PROFILE, "settings": settings, "methods": list(methods),
}, indent=2) + "\n")
print(f"{'method':8s} {'modal h':>10s} {'mean h':>10s} {'median h':>10s} {'mean ISE':>12s}")
for row in summary_rows:
    print(f"{row['method']:8s} {row['modal_h']:10.5f} {row['mean_h']:10.5f} {row['median_h']:10.5f} {row['mean_ise']:12.6f}")
print(f"CSV data and settings saved to {run_directory}")


## Bandwidth-choice distribution figure

Overlaid bars show the two relative-frequency distributions. As in Figure 3.5, the horizontal positions are equally spaced geometric-grid indices, labeled with their bandwidth values. The optional dashed line shows the Gaussian population MISE optimum; the purple bars show the **sample-specific ISE choice**.
The title records the actual $n$, $M$, $N$, and $\lambda$, so a preview is distinguishable from the larger simulation. Every displayed figure is automatically exported to PDF; a PNG is also saved for inspection.
Source for the optional reference: [exact Gaussian MISE helper](kde_tools/bandwidths.py).

In [ ]:
colors = {"L2": "#D6843B", "ISE": "#9B73C5"}
labels = {"L2": r"$L^2$ spectral", "ISE": "ISE-optimal grid bandwidths"}
grid_positions = np.arange(len(h_grid))
fig, ax = plt.subplots(figsize=(13.5, 6.5))
for name in methods:
    ax.bar(grid_positions, frequencies[name], width=0.86, color=colors[name],
           alpha=0.48, edgecolor="white", linewidth=0.35, label=labels[name])
if show_mise_reference:
    mise_bandwidth = np.sqrt(settings["sigma2"]) * gaussian_mise_oracle_bandwidth(settings["sample_size"])
    if h_grid[0] <= mise_bandwidth <= h_grid[-1]:
        mise_position = np.interp(np.log(mise_bandwidth), np.log(h_grid), grid_positions)
        ax.axvline(mise_position, color="#1f77b4", linestyle="--", linewidth=2.0,
                   label=f"Gaussian MISE optimum: h={mise_bandwidth:.3f}")
ticks = np.unique(np.linspace(0, len(h_grid) - 1, 7, dtype=int))
ax.set_xticks(ticks, [f"{h_grid[index]:.3g}" for index in ticks])
ax.set_xlim(-0.7, len(h_grid) - 0.3)
ax.set_ylim(0, 1.08 * max(np.max(values) for values in frequencies.values()))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
ax.set_xlabel("selected bandwidth h", fontsize=13)
ax.set_ylabel("relative frequency", fontsize=13)
ax.set_title(
    "Bandwidth-choice distributions: Gaussian data\n"
    + rf"$n={settings['sample_size']}$, $M={repetitions}$, $N={len(h_grid)}$, "
    + rf"$S=\exp(-\lambda D)$, $\lambda={settings['lambda_value']:g}$ ({RUN_PROFILE})",
    fontsize=15, pad=18,
)
ax.grid(axis="y", color="#DDE5EC", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#B8C5D1")
ax.tick_params(labelsize=11, colors="#304967")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False, fontsize=10.5)
fig.tight_layout()
png_output_path = image_output_directory / f"{plot_stem}.png"
fig.savefig(png_output_path, dpi=160, bbox_inches="tight")
plt.show()  # The shared automatic exporter saves the PDF before display.
print(f"PNG: {png_output_path}")
print(f"PDF directory: {pdf_output_directory}")

## Separate bandwidth-choice distribution plots

Each distribution from the overlaid figure above is plotted again in its own figure, in the same method order and colors. Both plots use the same bandwidth positions and vertical limits as the overlaid figure. They reuse the existing `frequencies`; no new simulation is run.

The optional Gaussian MISE reference is repeated in each plot. Each figure is displayed below and exported as a PNG and, through the shared automatic exporter, a PDF.


In [ ]:
# Reuse the distributions and colors from the overlaid figure above.
separate_grid_positions = np.arange(len(h_grid))
separate_ticks = np.unique(np.linspace(0, len(h_grid) - 1, 7, dtype=int))
separate_y_max = 1.08 * max(np.max(values) for values in frequencies.values())
separate_mise_position = None
if show_mise_reference:
    separate_mise_bandwidth = np.sqrt(settings["sigma2"]) * gaussian_mise_oracle_bandwidth(settings["sample_size"])
    if h_grid[0] <= separate_mise_bandwidth <= h_grid[-1]:
        separate_mise_position = np.interp(
            np.log(separate_mise_bandwidth), np.log(h_grid), separate_grid_positions
        )

for name in methods:
    separate_fig, separate_ax = plt.subplots(figsize=(13.5, 6.5))
    separate_ax.bar(
        separate_grid_positions, frequencies[name], width=0.86,
        color=colors[name], alpha=0.85, edgecolor="white", linewidth=0.35,
        label=labels[name],
    )
    if separate_mise_position is not None:
        separate_ax.axvline(
            separate_mise_position, color="#1f77b4", linestyle="--", linewidth=2.0,
            label=f"Gaussian MISE optimum: h={separate_mise_bandwidth:.3f}",
        )
    separate_ax.set_xticks(
        separate_ticks, [f"{h_grid[index]:.3g}" for index in separate_ticks]
    )
    separate_ax.set_xlim(-0.7, len(h_grid) - 0.3)
    separate_ax.set_ylim(0, separate_y_max)
    separate_ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    separate_ax.set_xlabel("selected bandwidth h", fontsize=13)
    separate_ax.set_ylabel("relative frequency", fontsize=13)
    separate_ax.set_title(
        f"Bandwidth-choice distribution: {labels[name]}\n"
        + rf"Gaussian data: $n={settings['sample_size']}$, $M={repetitions}$, "
        + rf"$N={len(h_grid)}$, $\lambda={settings['lambda_value']:g}$ ({RUN_PROFILE})",
        fontsize=15, pad=18,
    )
    separate_ax.grid(axis="y", color="#DDE5EC", linewidth=0.8)
    separate_ax.set_axisbelow(True)
    for side in ("top", "right"):
        separate_ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        separate_ax.spines[side].set_color("#B8C5D1")
    separate_ax.tick_params(labelsize=11, colors="#304967")
    separate_ax.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.15),
        ncol=2, frameon=False, fontsize=10.5,
    )
    separate_fig.tight_layout()
    separate_png_output_path = image_output_directory / f"{plot_stem}_{name.lower()}.png"
    separate_fig.savefig(separate_png_output_path, dpi=160, bbox_inches="tight")
    plt.show()  # The shared automatic exporter saves this figure as a PDF.
    plt.close(separate_fig)
    print(f"PNG ({name}): {separate_png_output_path}")

print(f"PDF directory: {pdf_output_directory}")
